# 🚦 AI-Based Traffic Violation Detection with License Plate Recognition

**Hệ thống phát hiện vi phạm giao thông và nhận diện biển số xe sử dụng YOLOv8 + OCR**

### 👑 Cách tiếp cận tối ưu:
- **YOLOv8n**: Phát hiện phương tiện (xe máy, ô tô, xe tải, bus) + màu sắc
- **light_traffic.pt + HSV**: Phát hiện đèn giao thông
- **Virtual Stop Line**: Xe vượt vạch dừng khi đèn đỏ → VI PHẠM
- **Centroid Tracking**: Theo dõi xe qua các frame
- **license_plate_detector.pt + OCR**: Nhận diện biển số xe
- **helmet.pt**: Phát hiện vi phạm (không mũ, điện thoại, chở quá người)

---
## ⚙️ 1. Cài đặt môi trường & Kết nối Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/TrafficAI'
for sub in ['models','results','videos','images']:
    os.makedirs(f'{DRIVE_BASE}/{sub}', exist_ok=True)
print(f'✅ Drive mounted! Working: {DRIVE_BASE}')

In [ ]:
!pip install -q ultralytics opencv-python fast-plate-ocr numpy pillow matplotlib tqdm
print('✅ Dependencies installed!')

In [ ]:
MODEL_DIR = f'{DRIVE_BASE}/models'
def check_models():
    required = ['license_plate_detector.pt', 'yolov8n.pt', 'vehicle_color_n_cls.pt', 'helmet.pt', 'light_traffic.pt']
    missing = [m for m in required if not os.path.exists(f'{MODEL_DIR}/{m}')]
    if 'yolov8n.pt' in missing:
        from ultralytics import YOLO
        YOLO('yolov8n.pt')
        import shutil
        shutil.move('yolov8n.pt', f'{MODEL_DIR}/yolov8n.pt')
        missing.remove('yolov8n.pt')
        print('✅ yolov8n.pt downloaded')
    if not missing:
        print('✅ All models ready!')
    else:
        print(f'❌ Missing: {missing}')
        print(f'📤 Upload to: {MODEL_DIR}/')
    return len(missing) == 0
check_models()

---
## 🧠 2. Khởi tạo Models & Functions

In [ ]:
import cv2, numpy as np, time, re, os, base64
from datetime import datetime
from tqdm.notebook import tqdm
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt
from ultralytics import YOLO
from fast_plate_ocr import LicensePlateRecognizer as FastPlateOCR
from google.colab import files

print('✅ Libraries imported!')

In [ ]:
# === Load Models ===
yolo_plate = YOLO(f'{MODEL_DIR}/license_plate_detector.pt')
yolo_vehicle = YOLO(f'{MODEL_DIR}/yolov8n.pt')

try:
    color_model = YOLO(f'{MODEL_DIR}/vehicle_color_n_cls.pt')
    print('✅ Color model')
except:
    color_model = None

try:
    helmet_model = YOLO(f'{MODEL_DIR}/helmet.pt')
    print('✅ Violation model')
except:
    helmet_model = None

try:
    light_model = YOLO(f'{MODEL_DIR}/light_traffic.pt')
    print('✅ Traffic light model')
except:
    light_model = None

ocr = FastPlateOCR(hub_ocr_model='cct-s-v2-global-model', device='auto')
print('✅ OCR loaded')
print('\n🟢 All models ready!')

In [ ]:
COLOR_NAMES = ['beige','black','blue','brown','gold','green','grey','orange','pink','purple','red','silver','tan','white','yellow']
VEHICLE_CLASSES = {2:'car', 3:'motorcycle', 5:'bus', 7:'truck'}
VIOL_COLORS = {'QUA_TAI':(0,0,255), 'KHONG_MU':(0,165,255), 'DUNG_DT':(255,0,255), 'VUOT_DEN':(0,0,255)}

def detect_vehicles(image, conf=0.25):
    r = yolo_vehicle.predict(image, classes=list(VEHICLE_CLASSES.keys()), conf=conf, verbose=False)[0]
    out = []
    if r.boxes is None: return out
    for b in r.boxes:
        x1,y1,x2,y2 = map(int, b.xyxy[0])
        vtype = VEHICLE_CLASSES.get(int(b.cls[0]), 'vehicle')
        color = 'unknown'
        if color_model:
            crop = image[y1:y2, x1:x2]
            if crop.size > 0:
                try:
                    cr = color_model.predict(crop, verbose=False)[0]
                    if hasattr(cr,'probs') and cr.probs is not None:
                        idx = cr.probs.top1
                        color = COLOR_NAMES[idx] if idx < len(COLOR_NAMES) else 'unknown'
                except: pass
        out.append({'bbox':(x1,y1,x2,y2), 'vtype':vtype, 'color':color, 'conf':float(b.conf[0])})
    return out

def detect_plates(image, conf=0.1):
    r = yolo_plate.predict(image, conf=conf, verbose=False)[0]
    out = []
    if r.boxes is None: return out
    for b in r.boxes:
        x1,y1,x2,y2 = map(int, b.xyxy[0])
        h,w = image.shape[:2]
        out.append({'img':image[max(0,y1):min(h,y2), max(0,x1):min(w,x2)], 'bbox':(x1,y1,x2,y2)})
    return out

def extract_text(img):
    if img is None or img.size == 0: return ''
    try:
        raw = ocr.run(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        if hasattr(raw,'text'): s = raw.text
        elif isinstance(raw, str): s = raw
        elif isinstance(raw, list) and len(raw)>0:
            s = raw[0].text if hasattr(raw[0],'text') else str(raw[0])
        else: s = str(raw) if raw else ''
        for kw in ['PREDICTION','PLATE','CHARPROBS','CHARS','REGION','UNITEDKINGDOM','VIETNAM','NONE','PROB']:
            s = re.sub(kw, '', s, flags=re.IGNORECASE)
        s = re.sub(r'[^A-Z0-9\-.]', '', s.upper())
        m = re.search(r'(\d{1,2}[A-Z]{1,2}[-\d\.]*\d+)', s)
        if m: return m.group(1)
        m = re.search(r'([A-Z]{1,3}[0-9]{1,4}[A-Z]{0,3})', s)
        if m: return m.group(1)
        m = re.search(r'([A-Z0-9]{4,})', s)
        return m.group(1) if m else s
    except: return ''

def match_plates(vehicles, plates):
    matched = []
    for v in vehicles:
        vx1,vy1,vx2,vy2 = v['bbox']
        for p in plates:
            px1,py1,px2,py2 = p['bbox']
            if px1 >= vx1 and px2 <= vx2 and py1 >= vy1 and py2 <= vy2:
                t = extract_text(p['img'])
                if t:
                    matched.append({'vehicle_bbox':v['bbox'], 'plate_text':t, 'vtype':v['vtype'], 'color':v['color']})
                break
    return matched

def detect_violations(image):
    out = []
    if helmet_model is None: return out
    r = helmet_model.predict(image, conf=0.15, iou=0.45, agnostic_nms=True, verbose=False)
    if not r or r[0].boxes is None: return out
    for b in r[0].boxes:
        x1,y1,x2,y2 = map(int, b.xyxy[0])
        cn = r[0].names[int(b.cls[0])].strip().lower()
        c = float(b.conf[0])
        vt = None
        if 'more_than' in cn and c >= 0.35: vt = ('QUA_TAI', 'Xe chở quá 2 người')
        elif any(x in cn for x in ['without_helmet','no_helmet','w/o_helmet']) and c >= 0.15: vt = ('KHONG_MU', 'Không đội mũ bảo hiểm')
        elif any(x in cn for x in ['using_mobile','phone','mobile']) and c >= 0.15: vt = ('DUNG_DT', 'Sử dụng điện thoại')
        if vt: out.append({'type':vt[0], 'details':vt[1], 'bbox':(x1,y1,x2,y2), 'conf':c})
    return out

def detect_red_light(image):
    is_red = False; boxes = []
    if light_model is not None:
        for r in light_model.predict(image, conf=0.15, verbose=False):
            if not hasattr(r,'boxes') or r.boxes is None: continue
            for b in r.boxes:
                n = r.names[int(b.cls[0])].lower()
                if 'trafficlight' in n:
                    boxes.append(tuple(map(int, b.xyxy[0])))
                    if 'red' in n: is_red = True
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    h,s,v = cv2.split(hsv)
    bright = (v>=80) & (s>=60)
    red = int(np.count_nonzero(((h<=10)|(h>=170)) & bright))
    green = int(np.count_nonzero((h>=40)&(h<=95)&bright))
    if red > green*1.5 and red > 50: is_red = True
    elif green > red*1.5 and green > 50: is_red = False
    return is_red, boxes

def draw_frame(img, vehicles, matched, violations, rv, is_red, light_boxes, stop_line_y):
    frame = img.copy()
    h,w = frame.shape[:2]
    if stop_line_y:
        cv2.line(frame, (0,stop_line_y), (w,stop_line_y), (255,255,0), 3)
        cv2.putText(frame, 'VACH DUNG', (10,stop_line_y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)
    for bx in light_boxes:
        x1,y1,x2,y2 = bx
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,0,255) if is_red else (0,255,0), 2)
    pl = {tuple(m['vehicle_bbox']):m['plate_text'] for m in matched}
    rv_set = {rv2['bbox'] for rv2 in rv}
    for v in vehicles:
        x1,y1,x2,y2 = v['bbox']
        bk = v['bbox']
        c = (0,0,255) if bk in rv_set else (0,255,0)
        cv2.rectangle(frame, (x1,y1), (x2,y2), c, 2)
        cv2.putText(frame, f"{v['color']} {v['vtype']}", (x1,y1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.45, c, 1)
        plate = pl.get(bk, '')
        if plate: cv2.putText(frame, plate, (x1,y2+20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
        if bk in rv_set: cv2.putText(frame, 'VUOT DEN DO!', (x1,y1-25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 2)
    for viol in violations:
        vx1,vy1,vx2,vy2 = viol['bbox']
        col = VIOL_COLORS.get(viol['type'], (0,165,255))
        cv2.rectangle(frame, (vx1,vy1), (vx2,vy2), col, 2)
        lbl = f"{viol['type']} {viol['conf']*100:.0f}%"
        (tw,th),_ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
        cv2.rectangle(frame, (vx1, vy1-th-4), (vx1+tw+6, vy1), col, -1)
        cv2.putText(frame, lbl, (vx1+3, vy1-2), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1)
    st = f"{'DEN DO' if is_red else 'DEN XANH'} | {len(vehicles)} xe | {len(matched)} bien | {len(rv)} vuot"
    cv2.putText(frame, st, (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255) if is_red else (0,200,0), 2)
    return frame

def centroid(box):
    return ((box[0]+box[2])//2, (box[1]+box[3])//2)

def track_vehicles(current, prev, max_dist=100):
    new, used = [], set()
    active = [t for t in prev if time.time()-t['last_seen'] < 1.0]
    for v in current:
        vc = centroid(v['bbox'])
        best, bd = None, float('inf')
        for i, pt in enumerate(active):
            if i in used: continue
            d = np.sqrt((vc[0]-pt['centroid'][0])**2 + (vc[1]-pt['centroid'][1])**2)
            if d < bd and d < max_dist: bd, best = d, i
        if best is not None:
            used.add(best)
            t = active[best]; t['bbox']=v['bbox']; t['centroid']=vc; t['last_seen']=time.time()
            new.append(t)
        else:
            new.append({'bbox':v['bbox'], 'centroid':vc, 'vtype':v['vtype'], 'color':v['color'], 'conf':v['conf'],
                        'last_seen':time.time(), 'crossed':False, 'track_id':int(time.time()*1000)%100000})
    return new

print('✅ All functions ready!')

---
## 🖼️ 3. XỬ LÝ ẢNH

Upload ảnh để phát hiện xe, biển số, màu sắc, vi phạm.

In [ ]:
def process_image(image_path):
    img = cv2.imread(image_path)
    if img is None: print(f'❌ Không đọc được: {image_path}'); return
    print(f'📷 {os.path.basename(image_path)} ({img.shape[1]}x{img.shape[0]})')
    t0 = time.time()
    vehicles = detect_vehicles(img)
    plates = detect_plates(img); matched = match_plates(vehicles, plates)
    violations = detect_violations(img)
    is_red, _ = detect_red_light(img)
    rv = []
    if is_red:
        h = img.shape[0]
        for v in vehicles:
            if v['bbox'][3] > h*0.6: rv.append({'bbox':v['bbox'], 'vtype':v['vtype'], 'conf':v['conf']})
    result = draw_frame(img, vehicles, matched, violations, rv, is_red, [], None)
    print(f'⏱️ {time.time()-t0:.2f}s | 🚗 {len(vehicles)} | 🏷️ {len(matched)} | 🚨 {len(violations)+len(rv)}')
    plt.figure(figsize=(14,10))
    plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.tight_layout(); plt.show()
    cv2.imwrite(f'{DRIVE_BASE}/results/result_{os.path.basename(image_path)}', result)
    return result

uploaded = files.upload()
for fname in uploaded: process_image(fname)

---
## 📹 4. XỬ LÝ VIDEO REALTIME

Video được stream trực tiếp lên HTML. Xem kết quả ngay khi xử lý!

In [ ]:
# === HTML UI: Realtime Video + Log ===
REALTIME_HTML = '''
<div style="display:flex;flex-direction:column;align-items:center;background:#1e1e1e;padding:15px;border-radius:8px;width:870px">
  <div><img id="vf" src="" width="850" style="border-radius:4px"/></div>
  <div style="width:850px;margin-top:15px">
    <h4 style="color:#ff4d4d;margin:5px 0;font-family:sans-serif"> DANH SÁCH PHƯƠNG TIỆN VI PHẠM:</h4>
    <div id="logc" style="background:#000;color:#0f0;font-family:monospace;height:130px;overflow-y:scroll;padding:10px;border:1px solid #333;border-radius:4px;font-size:13px;line-height:1.5">
      [Hệ thống đang khởi động...]<br/>
    </div>
  </div>
</div>
<script>
function al(t){var d=document.getElementById('logc');d.innerHTML+=t+'<br/>';d.scrollTop=d.scrollHeight}
function uf(b){document.getElementById('vf').src='data:image/jpeg;base64,'+b}
</script>
'''

def process_video_realtime(video_path, output_path=None, stop_line_ratio=0.6, max_frames=500):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): print(f'❌ Không mở được: {video_path}'); return None
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    stop_y = int(h * stop_line_ratio)
    print(f'📹 {os.path.basename(video_path)}: {w}x{h} | Vạch dừng Y={stop_y}')
    writer = None
    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), min(fps,15), (w,h))
    clear_output(wait=True)
    display(HTML(REALTIME_HTML), display_id=True)
    prev_tracks, fc, proc = [], 0, 0
    stats = {'vehicles':0, 'plates':set(), 'violations':[], 'red_light':0, 'red_frames':0}
    is_red_stable = False
    red_hist = [False]*3
    last_log = {}
    pbar = tqdm(total=min(max_frames, total), desc='Realtime')
    while proc < max_frames:
        ret, frame = cap.read()
        if not ret: break
        fc += 1
        if fc % 2 != 0: continue
        vehicles = detect_vehicles(frame, conf=0.3)
        plates = detect_plates(frame); matched = match_plates(vehicles, plates)
        viol_list = detect_violations(frame)
        is_red, light_boxes = detect_red_light(frame)
        red_hist.append(is_red); red_hist.pop(0)
        if sum(red_hist) >= 2: is_red_stable = is_red
        if is_red_stable: stats['red_frames'] += 1
        prev_tracks = track_vehicles(vehicles, prev_tracks)
        rv = []
        for t in prev_tracks:
            if is_red_stable and t['bbox'][3] >= stop_y and not t['crossed']:
                t['crossed'] = True
                rv.append({'bbox':t['bbox'], 'vtype':t['vtype'], 'conf':t['conf']})
                stats['red_light'] += 1
                now = time.time()
                lk = f"{t['vtype']}_{t.get('track_id',0)}"
                if lk not in last_log or now - last_log[lk] > 2:
                    ts = datetime.now().strftime('%H:%M:%S')
                    html = f"<span style='color:#ff4d4d;'>[{ts}] VUOT DEN: <b>{t['vtype'].upper()}</b> (conf:{t['conf']:.2f})</span>"
                    display(HTML(f"<script>al('{html}');</script>"), display_id='la')
                    last_log[lk] = now
        stats['vehicles'] += len(vehicles)
        for m in matched: stats['plates'].add(m['plate_text'])
        for v in viol_list: stats['violations'].append(v)
        res = draw_frame(frame, vehicles, matched, viol_list, rv, is_red_stable, light_boxes, stop_y)
        cv2.putText(res, f'Frame:{proc}', (10,h-15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (180,180,180), 1)
        if writer: writer.write(res)
        _, buf = cv2.imencode('.jpg', res, [cv2.IMWRITE_JPEG_QUALITY, 70])
        display(HTML(f"<script>uf('{base64.b64encode(buf).decode()}');</script>"), display_id='va')
        proc += 1; pbar.update(1)
    pbar.close(); cap.release()
    if writer: writer.release()
    print('\n' + '='*50)
    print('📊 KẾT QUẢ')
    print('='*50)
    print(f'📹 {os.path.basename(video_path)}')
    print(f'📄 Frame xử lý: {proc}/{total}')
    print(f'🔴 Frame đèn đỏ: {stats["red_frames"]}')
    print(f'🚗 Xe: {stats["vehicles"]}')
    print(f'🏷️ Biển số: {", ".join(stats["plates"]) if stats["plates"] else "không"}')
    print(f'🚨 Vượt đèn đỏ: {stats["red_light"]}')
    print(f'🚨 Vi phạm khác: {len(stats["violations"])}')
    vc = {}; [vc.update({v["type"]:vc.get(v["type"],0)+1}) for v in stats["violations"]]
    for k,v in vc.items(): print(f'   {k}: {v}')
    if output_path: print(f'💾 Video kết quả: {output_path}')
    return stats

def process_video_from_drive(video_name, stop_line_ratio=0.6, max_frames=500):
    p = f'{DRIVE_BASE}/videos/{video_name}'
    if not os.path.exists(p):
        if os.path.exists(video_name): p = video_name
        else: print(f'❌ Không tìm thấy: {p}'); return None
    return process_video_realtime(p, f'{DRIVE_BASE}/results/processed_{video_name}', stop_line_ratio, max_frames)

def upload_and_process_video(stop_line_ratio=0.6, max_frames=300):
    u = files.upload()
    if not u: return None
    res = []
    for fn in u:
        print(f'\n📂 {fn}')
        r = process_video_realtime(fn, f'{DRIVE_BASE}/results/processed_{fn}', stop_line_ratio, max_frames)
        if r: res.append(r)
    return res

print('✅ Video REALTIME ready!')
print()
print('📌 Chạy: upload_and_process_video()')

In [ ]:
# === CHẠY TEST VIDEO REALTIME ===
result = upload_and_process_video(stop_line_ratio=0.6, max_frames=500)